# Mini2P batch preprocessing, mean-image export and review PDFs

Run cells one at a time, not **Run All**. Don't point the QC notebook's `run_dir` inside the batch's derivatives folder: a run folder with a `.phase.json` but no manifest is treated as finished and skipped. Paths, session range and options come from the `[batch]` and `[exports]` sections of `config/pipeline.toml`.

The batch cell finds every raw TIFF matching `source_substring` in the session range, numbers them `run-01`, `run-02`, ... by acquisition counter, and runs the full pipeline (`mini2p-preprocess`) for each pending one in a fresh process. Each run folder gets a `source_tiff.json` manifest and a `preprocess.log`. Completed runs are skipped; failed or interrupted ones are retried if `retry_incomplete_runs`; conflicting assignments stop the launch instead of overwriting.

In [ ]:
import logging
from pathlib import Path

from mini2p_preprocessing.batch import format_plan, load_batch_settings, plan_batch, run_batch

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)
CONFIG = Path("../config/pipeline.toml").resolve()
batch = load_batch_settings(CONFIG)
plan = plan_batch(batch)
print(f"Destriping: {batch.destripe}")
print(format_plan(plan))

In [ ]:
LAUNCH = False  # check the plan above, then set True and run this cell
if LAUNCH:
    # Re-plans from the manifests on disk, so rerunning after a failure only
    # launches runs that are still pending, and freezes a copy of the config.
    run_batch(batch, CONFIG)
else:
    print("Dry run only.")

## Export fixed-contrast mean images for every run

Saves raw float TIFFs, contrast PNGs, a red/green overlay and a settings JSON beside each `ops.npy`. Requires exactly `expected_runs` Suite2p outputs. Existing files are kept unless `overwrite = true` (`pdf_overwrite` for the PDFs). Set `run_subdir` to the folder between `run-XX/` and `suite2p/` that the batch used.

In [ ]:
from mini2p_preproc.exports import (
    export_all_mean_images,
    find_run_planes,
    load_export_settings,
    require_run_count,
    runs_with_both_settings_files,
    write_comparison_pdf,
    write_high_resolution_pdf,
)

exports = load_export_settings(CONFIG)
runs = find_run_planes(exports.subject_dir, exports.session_first, exports.session_last,
                       exports.run_subdir)
print(f"Found {len(runs)} Suite2p plane folders.")
require_run_count(runs, exports.expected_runs)
summary = export_all_mean_images(runs, exports.subject_dir, exports.session_first,
                                 exports.session_last, exports.contrast, exports.overwrite)
print("Summary:", summary)

## Review PDFs

One landscape page per run with green and red side by side, then a high-resolution PDF with green and red on separate portrait pages (contrast recomputed from `ops.npy`).

In [ ]:
both = runs_with_both_settings_files(runs)
print(f"Runs with both interactive and fixed contrast settings: {len(both)}")
for r in both:
    print(f"  ses-{r.session} {r.run_name}")

label = exports.session_label
comparison = write_comparison_pdf(
    runs, exports.subject_dir / f"meanImg_contrast_{label}.pdf", exports.contrast,
    f"Mini2P registered mean images, {label}", overwrite=exports.pdf_overwrite,
)
high_res = write_high_resolution_pdf(
    runs, exports.subject_dir / f"meanImg_high_resolution_{2 * len(runs)}_pages_{label}.pdf",
    exports.contrast, f"Mini2P high-resolution registered mean images, {label}",
    dpi=exports.high_resolution_dpi, overwrite=exports.pdf_overwrite,
)
print("Saved:", comparison)
print("Saved:", high_res)